In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import os

# TEST phenotypes

In [2]:
dpheno = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
dpheno = dpheno[dpheno['SET'] == 'TEST'].reset_index(drop=True)
print(dpheno["POP_SITE"].drop_duplicates().shape)
print(dpheno.shape)
dpheno["sample"] = dpheno["POP_SITE"]
dpheno.head()

(143,)
(2048, 8)


,Trait_name,POP_SITE,POP_ID,SITE_ID,SET,mean,sd,n,sample
0,Average_Ring_Density,1329_CH,1329,CH,TEST,476.575417,45.597986,8,1329_CH
1,Average_Ring_Density,1329_ML,1329,ML,TEST,480.010872,108.572664,3,1329_ML
2,Average_Ring_Density,1528_ML,1528,ML,TEST,514.502734,62.179284,3,1528_ML
3,Average_Ring_Density,1530_AC,1530,AC,TEST,553.321396,48.783433,14,1530_AC
4,Average_Ring_Density,1530_CH,1530,CH,TEST,526.616667,93.195567,6,1530_CH


In [3]:
dpheno[dpheno["n"] < 3].shape

(48, 9)

### Filtering out rows (pop_site x trait ) which have means obtained from less than 3 samples

In [4]:
dpheno = dpheno[dpheno['n']>2].reset_index(drop=True)
print(dpheno["POP_SITE"].drop_duplicates().shape)
print(dpheno.shape)


(142,)
(2000, 9)


# TRAIN phenotypes

In [5]:
tpheno = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
tpheno = tpheno[tpheno['SET'] == 'TRAIN'].reset_index(drop=True)
print(tpheno["POP_SITE"].drop_duplicates().shape)
print(tpheno.shape)
tpheno["sample"] = tpheno["POP_SITE"]

print(tpheno[tpheno["n"] < 3].shape)

### Filtering out rows (pop_site x trait ) which have means obtained from less than 3 samples
tpheno = tpheno[tpheno['n']>2].reset_index(drop=True)
print(tpheno["POP_SITE"].drop_duplicates().shape)
print(tpheno.shape)

(115,)
(1738, 8)
(108, 9)
(115,)
(1630, 9)


# Combining genetic offset sets

This is a list of all pop_sites x traits for which offsets were calculated, should correspond to the list in the TEST

In [6]:
traits = ["Height", "Biomass_Increment", "Biomass_Increment_1980", "Biomass_Increment_1985", 
                "Biomass_Increment_1990", "Biomass_Increment_1995", "Biomass_Increment_2000", 
                "Biomass_Increment_2005", "Biomass_Increment_2010", "Biomass_Increment_2015",
                "Average_Ring_Density", "DBH", "Rs", "Rl", "Rr","Rc"]

gardens = ["PR","ML","CH","AC"]


dsets = ["1000var5"]
#dsets = ["100","100LF","1000","1000LF","10000","lfmm","RDA","RDAcorrected"]
dsets

['1000var5']

In [8]:
D = []

for trait in traits:
    for dset in dsets:
        for garden in gardens:
            file_path = "../957B_garden_offset_run_var5/results/01_run_gradient_forest_" + garden + "_" + dset + "_" + trait + ".tsv"
            if os.path.exists(file_path):
                df_ = pd.read_csv(file_path, sep="\t", header=0)
                df_["marker"] = dset
                df_["trait"] = trait
                df_["garden"] = garden
                D.append(df_)
dD = pd.concat(D)
print(dD.shape)
dD.head()

(1992, 17)


,Trait_name,sample,SITE_ID,group,dPP,CMI,fallMT,now_dPP,now_CMI,now_fallMT,new_dPP,new_CMI,new_fallMT,offset,marker,trait,garden
0,Height,2209_PR,PR,West,56.200000,14.112083,-5.880183,0.086811,0.013422,-0.000731,0.072931,0.008084,0.013163,0.020352,1000var5,Height,PR
1,Height,325_PR,PR,Central,46.066667,59.986610,1.989890,0.018885,0.092677,0.015061,0.072931,0.008084,0.013163,0.100402,1000var5,Height,PR
2,Height,4351_PR,PR,Central,40.166667,38.597673,7.302088,-0.001850,0.064819,0.028152,0.072931,0.008084,0.013163,0.095057,1000var5,Height,PR
3,Height,4420_PR,PR,West,60.133333,-1.820773,-4.420879,0.102576,0.002729,0.001409,0.072931,0.008084,0.013163,0.032337,1000var5,Height,PR
4,Height,6805_PR,PR,East,49.466667,66.059520,2.622601,0.056524,0.096733,0.016342,0.072931,0.008084,0.013163,0.090211,1000var5,Height,PR


In [9]:
# List of pop_sites x traits = 141 - two missing (?)
dD_sub = dD[["sample","SITE_ID","offset","marker","trait"]].reset_index(drop=True)
print(dD_sub["sample"].drop_duplicates().shape)
print(dD_sub.shape)
dD_sub.head()

(140,)
(1992, 5)


,sample,SITE_ID,offset,marker,trait
0,2209_PR,PR,0.020352,1000var5,Height
1,325_PR,PR,0.100402,1000var5,Height
2,4351_PR,PR,0.095057,1000var5,Height
3,4420_PR,PR,0.032337,1000var5,Height
4,6805_PR,PR,0.090211,1000var5,Height


### Test

In [10]:
dD_merge = pd.merge(dD_sub, dpheno, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(dD_merge.shape)
dD_merge.head()

# There are rows with missing information - these are generally pop_sites x traits where phenotypes were measured on less than 3 samples  
# Removing these
dD_merge.loc[dD_merge["SET"].isna(),['sample']].drop_duplicates().shape
dD_merge.loc[dD_merge["SET"].isna(),['sample']].drop_duplicates()

dD_merge = dD_merge.dropna().reset_index(drop=True)
print(dD_merge.shape)

(1992, 12)
(1946, 12)


In [11]:
print(dD_merge["sample"].drop_duplicates().shape)

(139,)


### Train

In [12]:
tD_merge = pd.merge(dD_sub, tpheno, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(tD_merge.shape)
tD_merge.head()

tD_merge = tD_merge.dropna().reset_index(drop=True)
print(tD_merge.shape)
#tD_merge[tD_merge["SET"].isna()]
tD_merge.head()

(1992, 12)
(1622, 12)


,sample,SITE_ID,offset,marker,trait,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n
0,2209_PR,PR,0.020352,1000var5,Height,Height,2209_PR,2209.0,TRAIN,909.275258,298.708530,8.0
1,325_PR,PR,0.100402,1000var5,Height,Height,325_PR,325.0,TRAIN,1202.391145,197.315314,7.0
2,4351_PR,PR,0.095057,1000var5,Height,Height,4351_PR,4351.0,TRAIN,1311.931938,178.648122,12.0
3,4420_PR,PR,0.032337,1000var5,Height,Height,4420_PR,4420.0,TRAIN,732.439612,171.158517,7.0
4,6805_PR,PR,0.090211,1000var5,Height,Height,6805_PR,6805.0,TRAIN,1277.611112,147.869722,11.0


In [13]:
df_combo = pd.concat([dD_merge, tD_merge], ignore_index=True)
df_combo.head()
print(df_combo.shape)
df_combo[df_combo['SET'].isna()]
df_combo[df_combo['POP_ID'].isna()]

(3568, 12)


,sample,SITE_ID,offset,marker,trait,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n


In [14]:
df_combo["POP_ID"] = df_combo["POP_ID"].astype(int).astype(str)
df_combo["POP_ID"]

0       2209
1        325
2       4351
3       4420
4       6805
        ... 
3563    4277
3564    4351
3565    6941
3566    6969
3567    6983
Name: POP_ID, Length: 3568, dtype: object

## Adding group

Some pops do not have assigned groups : 6944 and 6937

In [15]:
dgroup = pd.read_csv("../DATA_intermediate/23_filter_clean_indiv_metrics.tsv", sep="\t", header=0)
dgroup["POP_ID"] = dgroup["POP"].astype(str).astype("object")
dgroup = dgroup[["POP_ID","group"]].drop_duplicates().reset_index(drop=True)
print(dgroup.shape)
dgroup.head()

add = pd.DataFrame({"POP_ID" : ["6937","6944"],
                   "group" : ["Central","Central"]})
#dgroup["POP_ID"].values
mgroup = pd.concat([dgroup, add], ignore_index=True)
print(mgroup.shape)
mgroup.tail()

#mgroup.to_csv("../METADATA/populations_genetic_groups.tsv", sep="\t", header=True, index=False)

(80, 2)
(82, 2)


,POP_ID,group
77,6969,West
78,995,West
79,7003,West
80,6937,Central
81,6944,Central


In [16]:
dm_combo = pd.merge(df_combo, mgroup, on = "POP_ID", how = "left")
dm_combo.head()

,sample,SITE_ID,offset,marker,trait,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group
0,2209_PR,PR,0.020352,1000var5,Height,Height,2209_PR,2209,TEST,980.525258,302.158899,4.0,West
1,325_PR,PR,0.100402,1000var5,Height,Height,325_PR,325,TEST,1239.891145,121.743290,8.0,Central
2,4351_PR,PR,0.095057,1000var5,Height,Height,4351_PR,4351,TEST,1423.598605,68.068593,4.0,Central
3,4420_PR,PR,0.032337,1000var5,Height,Height,4420_PR,4420,TEST,678.153897,154.272486,7.0,West
4,6805_PR,PR,0.090211,1000var5,Height,Height,6805_PR,6805,TEST,1353.974749,229.855027,7.0,East


In [17]:
dm_combo.loc[dm_combo["group"].isna(),'sample'].drop_duplicates()

Series([], Name: sample, dtype: object)

### Removing WI and ME

In [18]:
dc_combo = dm_combo[~dm_combo['group'].isin(["WI","ME"])].reset_index(drop=True)
print(dc_combo.shape)

(3452, 13)


In [19]:
dc_combo.to_csv("01_garden_compare_separate.tsv", sep = "\t", header=True, index=False)

# END